## 1. Установка необходимых библиотек

In [ ]:
!pip install --upgrade featuretools >> None
!pip install --upgrade lightgbm >> None

## 2. Импорт библиотек и настройка

In [ ]:
import pandas as pd
import numpy as np

import optuna

from sklearn.model_selection import train_test_split

import gc

import lightgbm as lgb

import warnings

## 3. Загрузка данных

In [ ]:
warnings.filterwarnings("ignore")

transaction_file = '/kaggle/input/dataset-generated/dataset_generated_with_cats.csv'
dataset_generated_with_cats = pd.read_csv(transaction_file)

In [ ]:
target_file = '/kaggle/input/alfa-challenge/train.pa'
df_target = pd.read_parquet(target_file)

In [ ]:
print("Данные успешно загружены.")

## 4. Подготовка данных

In [ ]:
target = df_target[['client_num', 'target']]

data_for_model = dataset_generated_with_cats.merge(target, on='client_num', how='inner')
print("Признаки и целевая переменная объединены для модели.")

cat_features = data_for_model.select_dtypes(include=['object', 'category']).columns.tolist()

X = data_for_model.drop(['client_num', 'target'], axis=1)
y = data_for_model['target']

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Данные разделены на обучающую и валидационную выборки.")

## 5. Расчет весов классов

In [ ]:
unique_classes = np.sort(y.unique())
class_counts = y_train.value_counts()
total_samples = len(y_train)
class_weights = {c: total_samples / (len(unique_classes) * class_counts[c]) for c in unique_classes}
print("Веса классов рассчитаны.")

def map_class_weights(y_labels, class_weights):
    return y_labels.map(class_weights).values

class WMAEMetric:
    def get_final_error(self, error, weight):
        return error / weight

    def is_max_optimal(self):
        return False

    def evaluate(self, approxes, target, weight):
        approx = approxes[0]
        target = np.array(target)
        weight = np.ones_like(target) if weight is None else np.array(weight)
        error = np.sum(weight * np.abs(target - approx))
        return error, np.sum(weight)
        
print("Кастомная метрика WMAE определена.")

weights_train = map_class_weights(y_train, class_weights)
weights_valid = map_class_weights(y_valid, class_weights)
print("Веса для выборок рассчитаны.")

## 6. Освобождение памяти и создание Dataset для LightGBM

In [ ]:
for col in cat_features:
    X_train[col] = X_train[col].astype('category')
    X_valid[col] = X_valid[col].astype('category')

categorical_feature_indices = [X_train.columns.get_loc(col) for col in cat_features]

lgb_train = lgb.Dataset(
    data=X_train,
    label=y_train,
    weight=weights_train,
    categorical_feature=cat_features,
    free_raw_data=False
)

lgb_valid = lgb.Dataset(
    data=X_valid,
    label=y_valid,
    weight=weights_valid,
    categorical_feature=cat_features,
    reference=lgb_train,
    free_raw_data=False
)

# Освобождение памяти
del X
del y

del data_for_model

del X_train
del X_valid
del y_train
del y_valid

gc.collect()

## 7. Оптимизация гиперпараметров с использованием Optuna

In [ ]:
print("\nОптимизация гиперпараметров с помощью Optuna...")

def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'mae',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'random_state': 42,
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.005, 0.5),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'num_leaves': trial.suggest_int('num_leaves', 2, 256),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 50),
        'min_child_samples': trial.suggest_int('min_child_samples', 1, 100),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.1, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.1, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 0, 15),
        'lambda_l1': trial.suggest_float('lambda_l1', 0.0, 15.0),
        'lambda_l2': trial.suggest_float('lambda_l2', 0.0, 15.0)
    }

    gbm = lgb.train(
       params,
       lgb_train,
       num_boost_round=10000,
       valid_sets=[lgb_valid],
       callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=True),
            lgb.log_evaluation(period=50)
       ]
    )

    preds = gbm.predict(lgb_valid.data, num_iteration=gbm.best_iteration)
    wmae = np.sum(weights_valid * np.abs(lgb_valid.label - preds)) / np.sum(weights_valid)
    return wmae

study = optuna.create_study(direction='minimize')

study.optimize(objective, timeout=10*60*60) 

print("Наилучшие гиперпараметры:")
print(study.best_params)

## 8. Обучение модели с лучшими гиперпараметрами

In [ ]:
best_params = study.best_params
best_params.update({
    'objective': 'regression',
    'metric': 'mae',
    'verbosity': -1,
    'random_state': 42
})

bst = lgb.train(
    best_params,
    lgb_train,
    num_boost_round=10_000_000,
    valid_sets=[lgb_valid],
    callbacks=[
        lgb.early_stopping(stopping_rounds=250, verbose=True),
        lgb.log_evaluation(period=50)
    ]
)


y_valid_pred = bst.predict(lgb_valid.data, num_iteration=bst.best_iteration)
wmae = np.sum(weights_valid * np.abs(lgb_valid.label - y_valid_pred)) / np.sum(weights_valid)
print('LightGBM Validation WMAE:', wmae)

## 9. Результаты обучения

In [ ]:
model_results = {
    'Model': ['LightGBM'],
    'Hyperparameters': [best_params],
    'Score (WMAE)': [wmae]
}

results_df = pd.DataFrame(model_results)

print("\nРезультаты:")
print(results_df)